# Phase 2 — Detection Baseline

**Goal:** Evaluate RF-DETR SiteSense vs YOLOv8n COCO on 10 sample frames.
Pick one detector. Commit findings to PROGRESS.md.

Run all cells top to bottom. Do not skip Cell 1.

In [ ]:
# Cell 1 — Repo sync (ALWAYS run first)
import os, sys

repo_path = "/kaggle/working/trailer-counter"
if not os.path.exists(repo_path):
    os.system("git clone https://github.com/Rutwik1000/trailer-counter.git " + repo_path)
os.chdir(repo_path)
os.system("git fetch origin")
os.system("git reset --hard origin/master")
os.system("git log --oneline -3")

if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

In [ ]:
# Cell 2 — Install dependencies
os.system("pip install -r requirements.txt -q")
for mod in list(sys.modules.keys()):
    if mod.startswith("src"):
        del sys.modules[mod]

In [ ]:
# Cell 3 — Verify environment
import torch
print("Python:", sys.version)
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

import cv2
import numpy as np
from src.detector import Detector
from src.preprocessor import apply_clahe
print("src.detector import OK")
print("src.preprocessor import OK")

In [ ]:
# Cell 4 — HuggingFace authentication
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

token = UserSecretsClient().get_secret("hf_token")
login(token=token, add_to_git_credential=False)
print("HF login OK")

In [ ]:
# Cell 5 — Download model weights (re-download if Kaggle session reset)
from huggingface_hub import hf_hub_download

os.makedirs("models", exist_ok=True)

rfdetr_dest = os.path.join("models", "rfdetr_construction.pth")
if os.path.exists(rfdetr_dest):
    print("Already exists: rfdetr_construction.pth")
else:
    hf_hub_download(
        repo_id="Zaafan/sitesense-weights",
        filename="rfdetr_construction.pth",
        local_dir="models/",
    )
    print("Downloaded: rfdetr_construction.pth")

assert os.path.exists(rfdetr_dest), "rfdetr_construction.pth not found — check HF repo/Kaggle secret"
print("RF-DETR weights present:", rfdetr_dest)
print("YOLOv8n: auto-downloaded by ultralytics on first detect() call")

In [ ]:
# Cell 6 — Load 10 sample frames (evenly distributed from 50)
import glob

all_paths = sorted(glob.glob(os.path.join(repo_path, "data", "frames", "*.jpg")))
assert len(all_paths) >= 10, f"Expected >=10 frames in data/frames/, found {len(all_paths)}"

# Evenly sample 10 frames from the full set
indices = [int(i * (len(all_paths) - 1) / 9) for i in range(10)]
sample_paths = [all_paths[i] for i in indices]

frames_raw = [cv2.imread(p) for p in sample_paths]
frames_enhanced = [apply_clahe(f) for f in frames_raw]

assert all(f is not None for f in frames_raw), "A frame failed to load"
print(f"Loaded {len(frames_raw)} frames: shape={frames_raw[0].shape}, dtype={frames_raw[0].dtype}")
print("CLAHE applied to all frames")

In [ ]:
# Cell 7 — YOLOv8n COCO detection on 10 frames
import matplotlib.pyplot as plt
import matplotlib.patches as patches

det_yolo = Detector(model_type="yolo", weights_path="yolov8n.pt", confidence_threshold=0.25)

yolo_results = [det_yolo.detect(f) for f in frames_enhanced]

total_yolo = sum(len(r) for r in yolo_results)
print(f"YOLOv8n — total detections across 10 frames: {total_yolo}")
for i, (path, dets) in enumerate(zip(sample_paths, yolo_results)):
    print(f"  Frame {i:02d} ({os.path.basename(path)}): {len(dets)} det")

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for idx, (ax, frame, dets, path) in enumerate(zip(axes.flatten(), frames_enhanced, yolo_results, sample_paths)):
    ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    for det in dets:
        x1, y1, x2, y2 = det["bbox"]
        ax.add_patch(patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=1.5, edgecolor="lime", facecolor="none"
        ))
        ax.text(x1, y1 - 3, f"{det['confidence']:.2f}", color="lime", fontsize=6)
    ax.set_title(f"Frame {idx:02d} — {len(dets)} det", fontsize=7)
    ax.axis("off")
plt.suptitle("YOLOv8n COCO (lime boxes)", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 8 — RF-DETR SiteSense detection on 10 frames
det_rfdetr = Detector(
    model_type="rfdetr",
    weights_path=os.path.join(repo_path, "models", "rfdetr_construction.pth"),
    confidence_threshold=0.25,
)

rfdetr_results = [det_rfdetr.detect(f) for f in frames_enhanced]

total_rfdetr = sum(len(r) for r in rfdetr_results)
print(f"RF-DETR SiteSense — total detections across 10 frames: {total_rfdetr}")
for i, (path, dets) in enumerate(zip(sample_paths, rfdetr_results)):
    print(f"  Frame {i:02d} ({os.path.basename(path)}): {len(dets)} det")

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for idx, (ax, frame, dets, path) in enumerate(zip(axes.flatten(), frames_enhanced, rfdetr_results, sample_paths)):
    ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    for det in dets:
        x1, y1, x2, y2 = det["bbox"]
        ax.add_patch(patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=1.5, edgecolor="red", facecolor="none"
        ))
        ax.text(x1, y1 - 3, f"{det['confidence']:.2f}", color="red", fontsize=6)
    ax.set_title(f"Frame {idx:02d} — {len(dets)} det", fontsize=7)
    ax.axis("off")
plt.suptitle("RF-DETR SiteSense (red boxes)", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 9 — Comparison summary
# -----------------------------------------------------------------------
# HUMAN ACTION: After reviewing the visualizations above, set detector_choice.
#   "rfdetr" — RF-DETR SiteSense boxes are tighter / more accurate on trucks
#   "yolo"   — YOLOv8n COCO is better, or RF-DETR errored / no detections
# -----------------------------------------------------------------------

print("=" * 60)
print("DETECTOR COMPARISON SUMMARY")
print("=" * 60)
print(f"YOLOv8n  COCO      : {total_yolo} total detections across 10 frames")
print(f"RF-DETR  SiteSense : {total_rfdetr} total detections across 10 frames")
print()
print("Evaluate by eye:")
print("  1. Truck/trailer bboxes correctly localized?")
print("  2. Few false positives (non-vehicle boxes)?")
print("  3. Few missed trucks?")
print()

# SET THIS based on your visual inspection of Cells 7 and 8
detector_choice = "rfdetr"   # <-- change to "yolo" if needed

print(f"DETECTOR CHOICE: {detector_choice.upper()}")
print("Cell 10 will commit this to PROGRESS.md.")
print("Claude Code will then update config/detector_choice.json.")

In [ ]:
# Cell 10 — Update PROGRESS.md and commit back to repo
import datetime

today = datetime.date.today().isoformat()
progress_path = os.path.join(repo_path, "docs", "PROGRESS.md")

with open(progress_path, "r") as f:
    content = f.read()

content = content.replace(
    "| Phase 2 — Detection Baseline | Not started |",
    f"| Phase 2 — Detection Baseline | Complete | {today} |",
)

progress_entry = (
    f"\n## {today} — Phase 2 complete\n\n"
    f"**Files created:** src/detector.py, tests/test_detector.py, notebooks/02_detection.ipynb, config/detector_choice.json\n"
    f"**Tests passing:** 13/13 non-skipped locally; 6 frame-dependent tests ran here on Kaggle\n"
    f"**YOLOv8n detections:** {total_yolo} across 10 frames\n"
    f"**RF-DETR detections:** {total_rfdetr} across 10 frames\n"
    f"**DETECTOR RECOMMENDATION: {detector_choice.upper()}**\n"
    f"**Reason:** [fill in: e.g. 'RF-DETR tighter bbox on trucks, fewer FP' or 'YOLOv8 better recall']\n"
    f"**Blocker:** None\n"
    f"**Next:** Phase 3 — Tracking + Zone (BoxMOT BotSort + loading zone polygon calibration)\n"
)

insert_marker = "---\n"
insert_pos = content.find(insert_marker) + len(insert_marker)
content = content[:insert_pos] + progress_entry + content[insert_pos:]

with open(progress_path, "w") as f:
    f.write(content)

print("PROGRESS.md updated")
print(progress_entry)

from kaggle_secrets import UserSecretsClient
gh_token = UserSecretsClient().get_secret("github_token")
os.system("git config --global credential.helper store")
with open(os.path.expanduser("~/.git-credentials"), "w") as f:
    f.write("https://x-token:" + gh_token + "@github.com\n")
os.system("git config user.email 'kaggle@trailer-counter'")
os.system("git config user.name 'Kaggle Executor'")
os.system("git add docs/PROGRESS.md")
os.system("git diff --stat HEAD")
os.system(f"git commit -m 'docs: phase 2 complete — detector recommendation: {detector_choice}'")
os.system("git push origin HEAD:master")
print("Pushed to master")